# 04 — Business Insights

**Objetivo:** Traduzir os dados em métricas e conclusões acionáveis para o negócio.

**KPIs analisados:**
- Receita total, ticket médio e volume de pedidos
- Taxa de itens faltando
- Top 10 produtos mais vendidos
- Receita e pedidos por região
- Ticket médio por faixa etária de cliente

In [ ]:
import sys
sys.path.append("..")

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.visualization import bar_chart

master      = pd.read_parquet("../data/processed/master.parquet")
products    = pd.read_parquet("../data/processed/products.parquet")
order_items = pd.read_parquet("../data/processed/order_items.parquet")

sns.set_theme(style="whitegrid")
print(f"Dados carregados: {master.shape[0]:,} pedidos")

## 1. KPIs executivos

In [ ]:
kpis = {
    "Total de Pedidos":        f"{master.shape[0]:,}",
    "Receita Total":           f"${master['order_amount'].sum():,.0f}",
    "Ticket Médio":            f"${master['order_amount'].mean():,.2f}",
    "Taxa de Itens Faltando":  f"{master['has_missing'].mean()*100:.1f}%",
    "Total de Clientes":       f"{master['customer_id'].nunique():,}",
    "Total de Motoristas":     f"{master['driver_id'].nunique():,}",
    "Regiões Atendidas":       f"{master['region'].nunique()}",
}

print("=" * 40)
print("         RESUMO DO NEGÓCIO")
print("=" * 40)
for k, v in kpis.items():
    print(f"  {k:<28} {v}")
print("=" * 40)

## 2. Receita e volume de pedidos por região

In [ ]:
region_summary = (
    master.groupby("region")
    .agg(
        total_orders=("order_id", "count"),
        total_revenue=("order_amount", "sum"),
        avg_ticket=("order_amount", "mean"),
        missing_rate=("has_missing", "mean"),
    )
    .round(2)
    .sort_values("total_revenue", ascending=False)
    .reset_index()
)

region_summary["missing_rate"] = (region_summary["missing_rate"] * 100).round(1).astype(str) + "%"
region_summary["total_revenue"] = region_summary["total_revenue"].apply(lambda x: f"${x:,.0f}")
region_summary["avg_ticket"]    = region_summary["avg_ticket"].apply(lambda x: f"${x:,.2f}")

display(region_summary)

## 3. Top 10 produtos mais pedidos

In [ ]:
top_products = (
    order_items
    .merge(products[["product_id", "product_name", "category", "price"]], on="product_id", how="left")
    .groupby(["product_name", "category"])
    .size()
    .reset_index(name="order_count")
    .sort_values("order_count", ascending=False)
    .head(10)
)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top_products["product_name"][::-1], top_products["order_count"][::-1], color="steelblue")
ax.set_title("Top 10 Produtos Mais Pedidos", fontsize=13, fontweight="bold")
ax.set_xlabel("Quantidade de Pedidos")
plt.tight_layout()
plt.savefig("../reports/figures/07_top_products.png", dpi=150)
plt.show()

display(top_products.reset_index(drop=True))

## 4. Ticket médio por faixa etária

In [ ]:
age_bins   = [18, 30, 45, 60, 100]
age_labels = ["18-29", "30-44", "45-59", "60+"]

master["age_group"] = pd.cut(master["customer_age"], bins=age_bins, labels=age_labels, right=False)

ticket_by_age = (
    master.groupby("age_group", observed=True)["order_amount"]
    .mean()
    .reset_index()
    .rename(columns={"order_amount": "avg_ticket"})
)

fig = bar_chart(
    ticket_by_age, x="age_group", y="avg_ticket",
    title="Ticket Médio por Faixa Etária do Cliente",
    xlabel="Faixa Etária", ylabel="Ticket Médio ($)", color="#4a90d9"
)
fig.savefig("../reports/figures/08_ticket_by_age.png", dpi=150)
plt.show()

display(ticket_by_age.assign(avg_ticket=ticket_by_age["avg_ticket"].apply(lambda x: f"${x:.2f}")))